In [4]:
import pandas as pd
from odf import opendocument
from odf.table import Table, TableRow, TableCell
from odf.text import P

def read_ods(file_path):
    """Liest eine .ods-Datei und gibt ein pandas DataFrame zurück."""
    doc = opendocument.load(file_path)
    tables = doc.getElementsByType(Table)
    if not tables:
        raise ValueError("Keine Tabellen in der ODS-Datei gefunden")

    sheet = tables[0]
    rows = sheet.getElementsByType(TableRow)

    header = [str(cell) for cell in rows[0].getElementsByType(TableCell)]
    data = []
    for row in rows[1:]:
        cells = row.getElementsByType(TableCell)
        data.append([str(cell) for cell in cells])

    return pd.DataFrame(data, columns=header), doc, sheet

def vergleiche_eautos_ods(eingabe_pfad, ausgabe_pfad, fabia_verkaufspreis, benzinpreis=2.20, strompreis=0.30):
    """
    Vergleicht mehrere E-Autos mit Fabia und fügt Ergebnisse der ODS-Datei hinzu.

    Args:
        eingabe_pfad: Pfad zur Eingabe-ODS (Spalten: Name, Kaufpreis, Verbrauch)
        fabia_verkaufspreis: Verkaufspreis des Fabia in €
        benzinpreis: Benzinpreis in €/l (Default: 2.20)
        strompreis: Strompreis in €/kWh (Default: 0.30)
    """
    df, doc, sheet = read_ods(eingabe_pfad)

    fabia_verbrauch = 4.5  # l/100km
    fabia_kosten_pro_km = (fabia_verbrauch / 100) * benzinpreis

    # Neue Spaltenüberschriften
    new_headers = [
        'Fabia_Verkaufspreis', 'Benzinpreis', 'Strompreis',
        'Fabia_Kosten_pro_km', 'Eauto_Kosten_pro_km', 'Ersparnis_pro_km',
        'Preisunterschied', 'Amortisation_km', 'Amortisation_Jahre_10k',
        'Amortisation_Jahre_20k', 'Lohnt_sich'
    ]

    # Header-Reihe aktualisieren
    header_row = sheet.getElementsByType(TableRow)[0]
    for header in new_headers:
        new_cell = TableCell()
        new_cell.addElement(P(text=str(header)))
        header_row.addElement(new_cell)

    # Datenreihen aktualisieren
    data_rows = sheet.getElementsByType(TableRow)[1:]
    for i, (_, row) in enumerate(df.iterrows()):
        eauto_name = row['Name']
        eauto_preis = float(row['Kaufpreis'])
        eauto_verbrauch = float(row['Verbrauch'])

        eauto_kosten_pro_km = (eauto_verbrauch / 100) * strompreis
        preis_differenz = eauto_preis - fabia_verkaufspreis
        ersparnis_pro_km = fabia_kosten_pro_km - eauto_kosten_pro_km

        if ersparnis_pro_km <= 0:
            amortisation_km = "NIE"
            amortisation_jahre_10k = "NIE"
            amortisation_jahre_20k = "NIE"
        else:
            amortisation_km = preis_differenz / ersparnis_pro_km
            amortisation_jahre_10k = amortisation_km / 10000
            amortisation_jahre_20k = amortisation_km / 20000

        new_values = [
            fabia_verkaufspreis, benzinpreis, strompreis,
            fabia_kosten_pro_km, eauto_kosten_pro_km, ersparnis_pro_km,
            preis_differenz, amortisation_km, amortisation_jahre_10k,
            amortisation_jahre_20k, "JA" if ersparnis_pro_km > 0 else "NEIN"
        ]

        data_row = data_rows[i]
        for value in new_values:
            new_cell = TableCell()
            new_cell.addElement(P(text=str(value)))
            data_row.addElement(new_cell)

    # Dokument speichern
    doc.save(ausgabe_pfad)
    print(f"Ergebnisse wurden zur geschrieben!")
    return df

# Beispielaufruf
if __name__ == "__main__":
    FABIA_VERKAUFSPREIS = 14000
    EINGABE_PFAD = "E Autos.ods"
    AUSGABE_PFAD = "E Autos vergleich.ods"

    vergleiche_eautos_ods(
        eingabe_pfad=EINGABE_PFAD,
        ausgabe_pfad= AUSGABE_PFAD,
        fabia_verkaufspreis=FABIA_VERKAUFSPREIS,
        strompreis=0.30
    )

Ergebnisse wurden zur geschrieben!
